# 01 — Chronos-2 do zero sobre KernelSynth → LMC-Synth

**Modelo publicado:** [baseline Chronos-2 / LMC-Synth](https://huggingface.co/mateuswiteck/tsfmxai-chronos2-lmc-synth-baseline) (reposit?rio privado; commit Hugging Face `2099eb8c737b40414fc1db6ff03477ba1947db93`). O modelo final est? na raiz; o checkpoint selecionado fica em `model_early_stopping/`.

Baseline de forecasting, **sem equation head e sem pesos pré-treinados**. Este
notebook contém treino completo, checkpoint/retomada, validação e teste. A execução
salva no Git é um **smoke test CPU com arquitetura pequena**, não o resultado do
baseline. `pilot` e `full` usam a arquitetura original de `amazon/chronos-2`, mas
baixam somente sua configuração e inicializam todos os pesos aleatoriamente.

## Por que usamos seis séries nesta baseline

O TimePFN publicado foi treinado com 160 canais. Neste experimento, fixamos **seis séries por exemplo** para o LMC-Synth e para a comparação futura com o nosso gerador. Seis séries ainda permitem dependências multivariadas e uma visualização legível das trajetórias e relações entre canais, exemplo por exemplo. Com 160 canais, métricas agregadas continuam possíveis, mas fica difícil inspecionar um caso individual e entender o que mudou. A redução também diminui o custo computacional por forward e o uso de memória. Consideramos seis séries suficientes para uma **primeira comparação controlada** dos métodos. Isso não demonstra que uma conclusão obtida com seis séries se estenda a 160.

Mantemos a mesma arquitetura Chronos-2, número de canais, tamanho de contexto e horizonte, batch efetivo e orçamento de atualizações nas futuras comparações entre geradores. Usar Chronos-2 em lugar do TimePFN isola melhor a diferença da distribuição sintética, que é o objeto da comparação. O Chronos começa com pesos aleatórios e esta baseline não usa equation head.

Outras adaptações também são explícitas: loss de quantis nativa do Chronos-2 em vez da MSE do TimePFN; seed do código oficial; validação sintética separada; orçamento fixo; e quatro valores Dirichlet/Weibull não divulgados pelos autores, tratados aqui como hipóteses locais. O protocolo não é uma reprodução exata do corpus original.

## Estimativa até o término do treinamento

O runner imprime `full_run_eta_hours` e `estimated_finish_utc` durante toda a execução. Primeiro, usa a taxa observada no KernelSynth para estimar também a etapa LMC: essa parte é provisória. Após começar o LMC, passa a usar sua taxa medida. A estimativa inclui espera por batches e atualizações observadas; validações, checkpoints, exportação e aquecimento do cache podem alterá-la. O piloto de dez atualizações por etapa verifica viabilidade e memória, mas não fornece um prazo confiável para o treino integral. O treino completo recalcula a previsão a cada cem atualizações e a mostra logo após a primeira.

Use `scripts/run_lmc_notebook.py` para ter saída em tempo real. Ao terminar, ele grava uma cópia do notebook executado em `outputs/chronos2_lmc/<run>/executed.ipynb`.

## Executar na máquina remota (Python 3.11)

Na raiz do repositório, com o ambiente GPU já usado na canonicalização ativo. Se faltar alguma importação, instale somente o pacote indicado pelo erro:
```bash
# Use o ambiente GPU existente; confirme as dependências antes de instalar algo.
python - <<'PY'
import importlib, torch
for name in ('numpy', 'scipy', 'sklearn', 'joblib', 'tqdm', 'gluonts',
             'pyarrow', 'threadpoolctl', 'nbformat', 'matplotlib',
             'pandas', 'safetensors', 'huggingface_hub',
             'transformers', 'accelerate', 'einops'):
    importlib.import_module(name)
assert torch.cuda.is_available(), 'Este Python não tem acesso à GPU'
print('Ambiente pronto:', torch.cuda.get_device_name(0))
PY
if [ ! -d experiments_local/chronos-forecasting/.git ]; then
  git clone https://github.com/amazon-science/chronos-forecasting.git experiments_local/chronos-forecasting
fi
git -C experiments_local/chronos-forecasting checkout 10afa9ebe016e514f9d7dc1aa873f66af57e116b
# O notebook inclui src/ no sys.path; não é preciso reinstalar o Chronos.
export TSFMXAI_LMC_MODE=pilot
export TSFMXAI_LMC_RUN=pilot-4070-01
python -u scripts/run_lmc_notebook.py
```
Se o checkout Chronos já existir, omita o clone; não descarte alterações locais.
A célula seguinte verifica sua revisão e baixa o checkout TimePFN fixado, se ausente.
O piloto compara `workers=0` e `workers=2` com pesos, dados e caches frios equivalentes.
Não o execute simultaneamente com o canonicalizer: a disputa pela GPU invalida a comparação.

Após analisar memória/tempo/perda do piloto, o treino integral usa:
```bash
export TSFMXAI_LMC_MODE=full
export TSFMXAI_LMC_RUN=full-4070-01
export TSFMXAI_LMC_WORKERS=2
export TSFMXAI_LMC_MICROBATCH=4
# Para retomar a MESMA configuração/diretório, defina TSFMXAI_LMC_RESUME=1.
python -u scripts/run_lmc_notebook.py
```
`full` não tem o limite do piloto: percorre as duas etapas inteiras. As saídas ficam
em `outputs/chronos2_lmc/<run>/`, ignoradas pelo Git. O checkpoint contém pesos,
Adam, scheduler, posição e RNG do PyTorch; só carregue checkpoints locais confiáveis.
Reexecutar sem `RESUME=1` não sobrescreve um treino existente. Para o notebook salvo
no Git, mantenha uma única célula de registro completo ao final.


In [2]:
import os, sys, json, subprocess, platform, importlib.metadata, hashlib
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from tsfmxai.datasets.lmc_synth import LMCSpec, TIMEPFN_REVISION, verify_timepfn
from tsfmxai.pipelines.chronos2_lmc import (
    TrainingSpec, scratch_model, train, CHRONOS_REVISION, CONFIG_REVISION,
)
UPSTREAM = ROOT / "experiments_local/TimePFN"
if not UPSTREAM.exists():
    UPSTREAM.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/egetaga/TimePFN.git", str(UPSTREAM)], check=True)
    subprocess.run(["git", "-C", str(UPSTREAM), "checkout", TIMEPFN_REVISION], check=True)
SOURCE_HASHES = verify_timepfn(UPSTREAM)
CHRONOS = ROOT / "experiments_local/chronos-forecasting"
assert subprocess.check_output(["git", "-C", str(CHRONOS), "rev-parse", "HEAD"], text=True).strip() == CHRONOS_REVISION
assert not subprocess.check_output(["git", "-C", str(CHRONOS), "status", "--porcelain", "--untracked-files=no", "--", "src"], text=True).strip()
sys.path.insert(0, str(CHRONOS / "src"))
import chronos, torch, numpy as np
assert Path(chronos.__file__).resolve().is_relative_to(CHRONOS.resolve()), "Restart kernel: another Chronos was imported"
print("TimePFN", TIMEPFN_REVISION, "Chronos", CHRONOS_REVISION)


TimePFN e11d5c2982b0e49b87873cbddcb7712641861268 Chronos 10afa9ebe016e514f9d7dc1aa873f66af57e116b


## Protocolo e limites da reprodução

Precedência: instruções deste experimento (Chronos do zero, seis canais) →
orientação dos autores de **06/11/2025** → código oficial fixado em **29/07/2025** →
artigo. Uma orientação qualitativa mais recente não fornece os números ausentes.

* O protocolo antigo de 6.000 passos, LR `1e-6`, contexto 512/horizonte 64 e um
  backward sem optimizer era provisório; não tinha fundamento encontrado no TimePFN.
* Código: 15.000 trajetórias LMC, comprimento 1.024; banco independente de 20.000
  séries univariadas; máximo de cinco kernels, contexto/horizonte 96/96, batch 16,
  uma passagem, stride temporal LMC 8 e independente 1, ruído multiplicativo
  normal com média 1 e desvio 0,1 nos inputs **e targets**. Adam e OneCycleLR.
* O banco independente é agrupado com stride `min(C//2,20)`; o loader original
  usa `[:-1]`. Com seis canais são 6.665 grupos × 833 janelas = 5.551.945 exemplos.
  LMC: 15.000 × 105 = 1.575.000. `drop_last` por etapa dá **346.996 + 98.437 =
  445.433 updates** e **7.126.928 exemplos utilizados**. São janelas sobrepostas,
  não milhões de trajetórias independentes. Esses números não são publicados para
  Chronos; são uma derivação explícita das regras de loader com seis canais.
* Artigo: 160 canais, seed 2023 e complemento independente ~25%; código: seed 2021,
  banco independente cujo loader produz 826.336 janelas com 160 canais, ~52,5% das
  1.575.000 LMC. Seguimos o código para seed/bancos, não forçamos 25%.
* O script usa um ciclo de LR; a recomendação posterior pede KernelSynth primeiro
  e LMC depois com LR menor. Adotamos **dois OneCycle**, picos `5e-4` e `1e-4`.
  **O segundo pico, a razão 1/5 e reiniciar o scheduler são escolhas locais**;
  os autores não publicaram esses números. Adam continua com seus estados entre etapas.
* **Não publicados:** `dirichlet_min`, `dirichlet_max`, `weibull_shape`,
  `weibul_scale`. Usamos explicitamente **0,1; 2; 2; 2**, herdados da preparação
  exploratória. Não são defaults científicos, nem resultado de busca. O autor
  confirma busca manual no regime de 160 canais, sem divulgar os valores finais.
  O comentário inicial da issue é de um usuário: seus números não são defaults dos autores.
* LMC usa `round(Weibull(shape)*scale+1)`, limitado a `[max(2,C//20), C]`:
  **2 a 6 latentes** aqui. Usa mistura convexa Dirichlet e GP com `eigh`.
  KernelSynth independente preserva o sampler sklearn/SVD original. Preservamos os
  33 kernels e duplicatas do banco oficial, seleção com reposição e soma/produto.
* A correção de julho na Weibull prevalece sobre a versão antiga. O wrapper fornece
  seed explícita ao `default_rng` LMC que originalmente usava entropia do sistema.
  Seed é derivada por trajetória/janela/split; número de workers não altera os dados.
  Trajetórias são float32, falhas não finitas interrompem o treino; não copiamos a
  imputação NaN do loader original. O retry de erro de álgebra linear permanece upstream.
* Arquitetura/loss: Chronos-2 original, inicializado aleatoriamente, com loss nativa
  de quantis; não a arquitetura TimePFN de ~8,5M parâmetros nem sua loss MSE.
  Registramos também MSE/MAE da mediana. Grupos `[16,96,6]` viram 96 linhas com
  16 group IDs distintos; microbatch 4 preserva quatro grupos inteiros e acumula
  quatro backwards antes de um único update. Todos os canais do mesmo exemplo interagem.
* FP32, sem clipping, sem weight decay, Adam `betas=(.9,.999), eps=1e-8`.
  OneCycle: `pct_start=.3`, cosine, `div_factor=25`, `final_div_factor=10000`,
  momentum/beta1 entre .85/.95. Microbatch 4 e workers 2 são escolhas para o hardware,
  não números do artigo; o script usa dez workers. Ajustar microbatch muda a ordem
  numérica/dropout, mas mantém exemplos e uma atualização por batch efetivo.
* Avaliação local: validação e teste sintéticos separados por seeds/bancos, 64
  exemplos por família no full, a cada 2.500 updates e ao final. A seleção por early stopping acompanha o LMC, enquanto o treino integral continua,
  para preservar orçamento fixo. O upstream valida em ETTh1 e pode parar antes.
  Não reproduzimos os resultados zero-shot reais publicados nem afirmamos equivalência.
* Cache preguiçoso de trajetórias: primeiro acesso gera, depois reutiliza. Não é
  geração de séries novas a cada update. Workers persistentes, spawn, uma thread
  BLAS por geração, prefetch 2 por worker, pin memory em CUDA; main process treina.
  Nenhum encoder/canonicalizador participa deste baseline.


In [4]:
MODE = os.environ.get("TSFMXAI_LMC_MODE", "smoke")
assert MODE in {"smoke", "pilot", "full"}
RUN = os.environ.get("TSFMXAI_LMC_RUN", f"{MODE}-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S"))
assert Path(RUN).name == RUN and RUN not in {".", ".."}
RESUME = os.environ.get("TSFMXAI_LMC_RESUME", "0") == "1"
# Deliberately required generator values: not upstream defaults.
PRODUCTION_DATA = LMCSpec(dirichlet_min=.1, dirichlet_max=2., weibull_shape=2., weibull_scale=2.)
DATA = PRODUCTION_DATA if MODE != "smoke" else replace(
    PRODUCTION_DATA, batch_size=2, independent_bank=8, correlated_bank=4)
SETTINGS = TrainingSpec(
    workers=int(os.environ.get("TSFMXAI_LMC_WORKERS", "2")) if MODE != "smoke" else 0,
    microbatch=int(os.environ.get("TSFMXAI_LMC_MICROBATCH", "4")) if MODE != "smoke" else 1,
    steps_per_stage={"smoke": 2, "pilot": 10, "full": None}[MODE],
    validation_batches=4 if MODE == "full" else 1,
    checkpoint_interval=2500, device="cpu" if MODE == "smoke" else "cuda")
if MODE != "smoke":
    assert torch.cuda.is_available(), "Pilot/full require CUDA; do not silently shrink the model"
torch.set_num_threads(1)
print("Mode:", MODE, "Run:", RUN)
print("Full budget:", {f: PRODUCTION_DATA.updates(f) for f in ("independent", "lmc")})
print("Actual settings:", asdict(SETTINGS))


Mode: full Run: lmc-full-12w-mb16-20261116
Full budget: {'independent': 346996, 'lmc': 98437}
Actual settings: {'independent_lr': 0.0005, 'lmc_lr': 0.0001, 'workers': 12, 'microbatch': 16, 'checkpoint_interval': 2500, 'validation_batches': 4, 'steps_per_stage': None, 'device': 'cuda', 'early_stopping_patience': 5, 'early_stopping_min_delta': 0.0}


## Dois modelos: com e sem early stopping

A mesma execu??o salva `model_early_stopping/` e `model_without_early_stopping/`.
A etapa independente sempre termina. Na etapa LMC, monitoramos a loss de valida??o
(splits de teste nunca participam da sele??o), a cada 2.500 updates e no final.
Paci?ncia: cinco avalia??es consecutivas sem melhora estrita; `min_delta=0`.
O melhor checkpoint anterior ao disparo ? congelado, enquanto o treinamento
continua at? completar o or?amento. N?o reiniciamos nem encurtamos o OneCycle.

`result.json` e o registro abaixo incluem: disparou ou n?o, motivo, update global,
update da etapa, hor?rio UTC, tempo acumulado at? o evento, posi??o do melhor
checkpoint e m?tricas de teste de ambos. O hor?rio da sele??o e o do disparo podem
ser diferentes. Na retomada, tempo acumulado exclui o per?odo desligado e trabalho
perdido ap?s o ?ltimo checkpoint. O estado de sele??o ? retomado junto do treino.

Se o crit?rio n?o disparar, registramos `budget_exhausted_without_trigger` e ainda
salvamos o melhor checkpoint LMC observado. No piloto h? s? uma valida??o por etapa:
portanto ele n?o tem avalia??es suficientes para disparar uma paci?ncia de cinco.
Este procedimento permite comparar os modelos sem duplicar o treinamento, mas
n?o economiza o tempo de treinamento da execu??o integral. Checkpoints antigos,
anteriores a esta pol?tica, n?o s?o compat?veis com a retomada desta vers?o.


In [6]:
RESULTS, HISTORIES = {}, {}
WORKER_ARMS = [0, SETTINGS.workers] if MODE == "pilot" and SETTINGS.workers != 0 else [SETTINGS.workers]
for workers in WORKER_ARMS:
    arm = f"workers-{workers}"
    settings = replace(SETTINGS, workers=workers)
    directory = ROOT / "outputs/chronos2_lmc" / RUN / arm
    model = scratch_model(seed=DATA.seed, tiny=MODE == "smoke")
    RESULTS[arm], HISTORIES[arm] = train(
        model, DATA, settings, UPSTREAM, directory / "cache", directory, resume=RESUME)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
RESULTS


{"progress_stage": "independent", "update": 12501, "total_updates_stage": 346996, "stage_eta_hours": 31.764453748816756, "full_run_eta_hours": 41.11226921294111, "eta_basis": "LMC extrapolated from independent stage", "estimated_finish_utc": "2026-11-18T16:39:09.743313+00:00"}
{"progress_stage": "independent", "update": 12600, "total_updates_stage": 346996, "stage_eta_hours": 9.52068956863899, "full_run_eta_hours": 12.323319142761036, "eta_basis": "LMC extrapolated from independent stage", "estimated_finish_utc": "2026-11-17T11:51:59.430863+00:00"}
{"progress_stage": "independent", "update": 12700, "total_updates_stage": 346996, "stage_eta_hours": 9.42911556643686, "full_run_eta_hours": 12.205618572794533, "eta_basis": "LMC extrapolated from independent stage", "estimated_finish_utc": "2026-11-17T11:45:05.767380+00:00"}
{"progress_stage": "independent", "update": 12800, "total_updates_stage": 346996, "stage_eta_hours": 9.397577296561407, "full_run_eta_hours": 12.165621547065946, "eta_b

## Diagnóstico de eficiência

O piloto usa caches separados inicialmente vazios e os mesmos IDs, pesos e shapes.
Compara duas execuções completas curtas, sem disputar a GPU. `next_batch_seconds`
mede espera exposta pelo consumidor, **não tempo de GPU stall**. Há sincronização
CUDA por update para delimitar os tempos, o que altera sobreposição; esta é uma
medição diagnóstica, não throughput máximo. `invocation_seconds` inclui validação,
teste, exportação, carregamento dos workers e checkpoint, mas exclui construir/baixar
o modelo. Não confundir esses denominadores.

A extrapolação abaixo é apenas o cenário de custo médio observado no piloto:
a cobertura do cache muda muito ao longo de centenas de milhares de updates.
Dez updates por etapa não estabilizam esse regime. Antes de comprometer a GPU por
dias, faça um piloto mais longo alterando `steps_per_stage`, confira a curva de
perda e repita a comparação em ordem inversa e com cache quente. Um profiler CUDA
ou Nsight seria necessário para atribuir gaps de GPU à entrada de dados. Não há
uma medição de stall real nesta execução, nem uma comparação offline/on-the-fly
concluída. O primeiro objetivo deste notebook é estabelecer o baseline LMC.


In [8]:
TIMING = {}
for arm, history in HISTORIES.items():
    TIMING[arm] = {}
    for family in ("independent", "lmc"):
        rows = [r for r in history if r["stage"] == family]
        seconds = np.array([r["next_batch_seconds"] + r["train_seconds"] for r in rows])
        TIMING[arm][family] = {
            "measured_updates": len(rows), "seconds_per_update": float(seconds.mean()),
            "exposed_input_wait_fraction": float(sum(r["next_batch_seconds"] for r in rows) / seconds.sum()),
            "scenario_full_hours_excluding_eval_and_save": None if MODE == "smoke" else float(seconds.mean() * PRODUCTION_DATA.updates(family) / 3600),
        }
if len(WORKER_ARMS) == 2:
    serial, parallel = [f"workers-{w}" for w in WORKER_ARMS]
    print("Pilot end-to-end speedup:", RESULTS[serial]["invocation_seconds"] / RESULTS[parallel]["invocation_seconds"])
if MODE == "smoke":
    print("CPU tiny smoke timing is NOT a GPU training estimate.")
TIMING


In [9]:
# REGISTRO COMPLETO DO EXPERIMENTO — configuração, proveniência, seeds,
# métricas, síntese científica e reprodução em uma única célula.
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
RECORD = {
    "title": "Chronos-2 scratch / KernelSynth -> LMC-Synth / six channels",
    "status": "executed smoke test; baseline NOT trained" if MODE == "smoke" else f"executed {MODE}",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "run": RUN, "mode": MODE, "initialization": "all random weights; config only from Hugging Face",
    "production_data": asdict(PRODUCTION_DATA), "executed_data": asdict(DATA),
    "training": asdict(SETTINGS), "worker_arms": WORKER_ARMS,
    "seed_policy": "2021 from run.py; SeedSequence(seed, split, family, trajectory/window ID); splits 0/1/2 disjoint",
    "model_config_revision": CONFIG_REVISION, "chronos_code_revision": CHRONOS_REVISION,
    "timepfn_revision": TIMEPFN_REVISION, "upstream_generator_sha256": SOURCE_HASHES,
    "repository_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "repository_status_at_execution": subprocess.check_output(["git", "-C", str(ROOT), "status", "--short"], text=True),
    "implementation_sha256": {p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in [
        "src/tsfmxai/datasets/lmc_synth.py", "src/tsfmxai/pipelines/chronos2_lmc.py",
        "src/tsfmxai/datasets/chronos2_adapter.py"]},
    "environment": {"python": sys.version, "platform": platform.platform(),
        "versions": {p: version(p) for p in ["numpy", "scipy", "scikit-learn", "torch", "transformers", "gluonts", "threadpoolctl", "chronos-forecasting"]},
        "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else None},
    "sources": {
        "paper": "https://arxiv.org/pdf/2502.16294 (v1: algorithm p2, generator p4, training p6, Appendix E p11)",
        "bibliography": "references/library.bib:taga2025timepfn",
        "code": "https://github.com/egetaga/TimePFN/tree/" + TIMEPFN_REVISION,
        "latest_author_guidance": "https://github.com/egetaga/TimePFN/issues/1#issuecomment-3498477701 (owner egetaga, 2025-11-06)",
    },
    "precedence": "user six-channel scratch Chronos decision > Nov 2025 author guidance > July 2025 code > paper",
    "published_or_code_defaults": {
        "banks": "20000 univariate KernelSynth + 15000 multivariate LMC; length1024; maxkernels5",
        "windows": "context96, horizon96, LMC stride8, independent time stride1; group stride min(C//2,20)",
        "batch_optimizer": "batch16, one pass, Adam(.9,.999,eps1e-8,wd0), peakLR5e-4, FP32, no clipping",
        "onecycle": "pct_start=.3, cosine, div25, final_div10000, beta1 .85-.95",
        "noise": "multiplicative N(1,.1) independently on context and future", "seed": 2021,
    },
    "not_published_local_choices": {
        "generator": "Dirichlet .1..2, Weibull shape2 scale2: exploratory choices, NOT identified original settings",
        "schedule": "second peakLR1e-4 and two separate OneCycles; author gives only smaller LR",
        "hardware": "microbatch4, workers2, spawn/persistent/prefetch2, pinned CPU tensors; torch/BLAS threads1",
        "evaluation": "disjoint synthetic validation/test, 4 full batches per family, shadow early stopping on LMC validation, patience5/min_delta0; save selected and full-budget models",
    },
    "divergences": [
        "Chronos random architecture/native quantile loss instead of TimePFN/MSE; six instead of160 channels",
        "Code seed2021 instead of paper2023; code bank/window counts instead of paper25% independent",
        "Author later recommends lower second-stage LR; replaces older single OneCycle with explicit local two-cycle schedule",
        "Latest Weibull implementation used; deterministic seed added to otherwise entropy-seeded LMC GP",
        "float32 cache; nonfinite samples fail rather than upstream NaN imputation; raw trajectories reused",
        "Synthetic holdouts rather than ETTh1 validation; fixed updates plus frozen early-stopping selection on LMC validation",
        "No published complete generator parameter values, generated corpus seeds, or exact newer two-stage schedule found",
    ],
    "full_budget": {f: {"windows": PRODUCTION_DATA.windows(f), "updates": PRODUCTION_DATA.updates(f)} for f in ("independent", "lmc")},
    "results": RESULTS, "timing": TIMING,
    "scientific_summary": "Smoke only validates execution, optimizer updates, checkpoint/export and held-out metrics; it cannot establish forecast quality. Pilot estimates local cost, not paper performance. Full is an adapted baseline, not an exact TimePFN reproduction.",
    "timing_limit": "Consumer wait is not GPU stall; synchronized update timing, cold-cache pilot extrapolation is not a reliable full-run ETA",
    "reproduction": "Use the existing GPU Python 3.11 environment, verify imports, and pin the Chronos source as above; set TSFMXAI_LMC_MODE=smoke|pilot|full and unique TSFMXAI_LMC_RUN; execute entire notebook with tsfmxai kernel. TSFMXAI_LMC_RESUME=1 resumes matching config.",
    "artifact_root": f"outputs/chronos2_lmc/{RUN} (ignored; model_early_stopping/, model_without_early_stopping/, latest.pt, telemetry.jsonl, result.json, cache/)",
}
print(json.dumps(RECORD, indent=2, ensure_ascii=False))


{
  "title": "Chronos-2 scratch / KernelSynth -> LMC-Synth / six channels",
  "status": "executed full",
  "timestamp_utc": "2026-11-17T12:21:40.813843+00:00",
  "run": "lmc-full-12w-mb16-20261116",
  "mode": "full",
  "initialization": "all random weights; config only from Hugging Face",
  "production_data": {
    "dirichlet_min": 0.1,
    "dirichlet_max": 2.0,
    "weibull_shape": 2.0,
    "weibull_scale": 2.0,
    "channels": 6,
    "length": 1024,
    "context": 96,
    "horizon": 96,
    "batch_size": 16,
    "independent_bank": 20000,
    "correlated_bank": 15000,
    "max_kernels": 5,
    "lmc_stride": 8,
    "noise_std": 0.1,
    "seed": 2021
  },
  "executed_data": {
    "dirichlet_min": 0.1,
    "dirichlet_max": 2.0,
    "weibull_shape": 2.0,
    "weibull_scale": 2.0,
    "channels": 6,
    "length": 1024,
    "context": 96,
    "horizon": 96,
    "batch_size": 16,
    "independent_bank": 20000,
    "correlated_bank": 15000,
    "max_kernels": 5,
    "lmc_stride": 8,
    "noi